## Lab 8: Data warehousing

## Step 1 — Usage & Manual Testing Instructions

We compare three methods for loading updated CPI data into DuckDB.

Run all cells top to bottom. The database is saved to `lab08.db`.

**After Step 2** (load `PCPI24M1.csv`): all three tables have 924 rows, 1947-01 to 2023-12.

**After Step 3** (load `PCPI25M2.csv`):

| Table | Rows | Why |
|-------|------|-----|
| `cpi_append` | 1861 | Entire new file inserted on top — duplicates accumulate |
| `cpi_trunc` | 937 | Table wiped and reloaded — clean, captures revisions |
| `cpi_inc` | 937 | Only the 13 new months added — revisions to old values ignored |

In [53]:
#Importing necessary packages
import duckdb
import pandas as pd

## Step 2

Use the data in the file PCPI24M1.csv to initialize your database. This file contains inflation information as available in January 2024.
Create a persistent database containing three tables: cpi_append, cpi_trunc and cpi_inc.

In [54]:
original_data = pd.read_csv("PCPI24M1.csv")
df = original_data
df["DATE"] = pd.to_datetime(df["DATE"], format="%Y:%m") #changing the date and time format
df

,DATE,CPI
0,1947-01-01,21.5
1,1947-02-01,21.6
2,1947-03-01,22.0
3,1947-04-01,22.0
4,1947-05-01,22.0
...,...,...
919,2023-08-01,306.3
920,2023-09-01,307.5
921,2023-10-01,307.6
922,2023-11-01,307.9


In [55]:
#creating a DuckDB database
file = "lab08.db"
con = duckdb.connect(file)

# adding the original data to the database
con.register("df", df)
con.execute("CREATE OR REPLACE TABLE original_data AS SELECT * FROM df")

# creating three working tables
con.execute("CREATE OR REPLACE TABLE cpi_append AS SELECT * FROM original_data")
con.execute("CREATE OR REPLACE TABLE cpi_trunc AS SELECT * FROM original_data")
con.execute("CREATE OR REPLACE TABLE cpi_inc AS SELECT * FROM original_data")

# show tables
con.execute("SHOW TABLES").fetchdf()

# close the connection
con.close()

## Step 3

Load additional inflation data contained in the file PCPI25M2.csv into your database. This file contains inflation data as available in February 2025. It contains additional observations and historical revisions with respect to the previous file.

In [56]:
new_data = pd.read_csv("PCPI25M2.csv")
new_data["DATE"] = pd.to_datetime(new_data["DATE"], format="%Y:%m")

### Append load method

In [57]:
def append_load(con, data):
    con.register("new_data", data)
    con.execute("INSERT INTO cpi_append SELECT * FROM new_data")

with duckdb.connect(file) as con:
    con.sql(
        "BEGIN TRANSACTION"
    )  # starting a transaction -- changes are synced once = improves performance
    append_load(con, new_data)
    con.sql("COMMIT")  # committing the transaction
    print(con.sql("SELECT * FROM cpi_append"))

┌─────────────────────┬────────┐
│        DATE         │  CPI   │
│      timestamp      │ double │
├─────────────────────┼────────┤
│ 1947-01-01 00:00:00 │   21.5 │
│ 1947-02-01 00:00:00 │   21.6 │
│ 1947-03-01 00:00:00 │   22.0 │
│ 1947-04-01 00:00:00 │   22.0 │
│ 1947-05-01 00:00:00 │   22.0 │
│ 1947-06-01 00:00:00 │   22.1 │
│ 1947-07-01 00:00:00 │   22.2 │
│ 1947-08-01 00:00:00 │   22.4 │
│ 1947-09-01 00:00:00 │   22.8 │
│ 1947-10-01 00:00:00 │   22.9 │
│          ·          │     ·  │
│          ·          │     ·  │
│          ·          │     ·  │
│ 2024-04-01 00:00:00 │  313.0 │
│ 2024-05-01 00:00:00 │  313.1 │
│ 2024-06-01 00:00:00 │  313.1 │
│ 2024-07-01 00:00:00 │  313.6 │
│ 2024-08-01 00:00:00 │  314.1 │
│ 2024-09-01 00:00:00 │  314.9 │
│ 2024-10-01 00:00:00 │  315.6 │
│ 2024-11-01 00:00:00 │  316.4 │
│ 2024-12-01 00:00:00 │  317.6 │
│ 2025-01-01 00:00:00 │  319.1 │
├─────────────────────┴────────┤
│ 1861 rows          2 columns │
│ (20 shown)                   │
└─────────

### Incremental load method

In [58]:
# We had to use AI-assisted code generation for the incremental load function, as it is more complex than the previous two. The function was something that we missed in the coding. It helps register the new data, finds the current maximum date in the cpi_inc table, and then inserts only the new rows with a date greater than the current maximum.

def incremental_load(con, data):
    con.register("new_data", data) #only adding new CPI data based on date
    
    #finding the current max date in the cpi_inc table (AI-assisted code generation)
    max_date_before = con.sql("SELECT MAX(DATE) AS max_date FROM cpi_inc").fetchone()[0]

    # Insert only rows with DATE greater than current max
    con.execute(f"INSERT INTO cpi_inc SELECT * FROM new_data WHERE DATE > '{max_date_before}'")

    # Fetch and return only the new rows that were added
    new_rows = con.sql(f"SELECT * FROM new_data WHERE DATE > '{max_date_before}' ORDER BY DATE").fetchdf()

    return new_rows


# Example usage (AI-assisted code generation)
with duckdb.connect("lab08.db") as con:
    # Run incremental load and get new rows
    new_rows_added = incremental_load(con, new_data)

    # Show what was actually added
    print("New data added by this load:")
    print(new_rows_added)

    # Show the updated cpi_inc table
    cpi_incremental = con.sql("SELECT * FROM cpi_inc ORDER BY DATE").fetchdf()
    print("\nFull cpi_inc table (after incremental load):")
    print(cpi_incremental)

New data added by this load:
         DATE    CPI
0  2024-01-01  309.8
1  2024-02-01  311.0
2  2024-03-01  312.1
3  2024-04-01  313.0
4  2024-05-01  313.1
5  2024-06-01  313.1
6  2024-07-01  313.6
7  2024-08-01  314.1
8  2024-09-01  314.9
9  2024-10-01  315.6
10 2024-11-01  316.4
11 2024-12-01  317.6
12 2025-01-01  319.1

Full cpi_inc table (after incremental load):
          DATE    CPI
0   1947-01-01   21.5
1   1947-02-01   21.6
2   1947-03-01   22.0
3   1947-04-01   22.0
4   1947-05-01   22.0
..         ...    ...
932 2024-09-01  314.9
933 2024-10-01  315.6
934 2024-11-01  316.4
935 2024-12-01  317.6
936 2025-01-01  319.1

[937 rows x 2 columns]


### Trunc and load

In [61]:
def trunc_and_load(con, data):
    con.register("new_data", data)
    # we simply truncate the table and load the new data
    con.sql("CREATE OR REPLACE TABLE cpi_trunc AS SELECT * FROM new_data")


with duckdb.connect(file) as con:
    trunc_and_load(con, new_data)
    print(con.sql("SELECT * FROM cpi_trunc").fetchdf())


          DATE    CPI
0   1947-01-01   21.5
1   1947-02-01   21.6
2   1947-03-01   22.0
3   1947-04-01   22.0
4   1947-05-01   22.0
..         ...    ...
932 2024-09-01  314.9
933 2024-10-01  315.6
934 2024-11-01  316.4
935 2024-12-01  317.6
936 2025-01-01  319.1

[937 rows x 2 columns]


## Discussion — How the three methods compare

**Append** is the simplest to implement but the most dangerous. Running it twice on the same data doubles your row count. It has no way to handle historical revisions — old values just pile up alongside new ones. It's really only safe when your source data is guaranteed to have no overlapping dates.

**Truncate & Load** is the safest for data correctness. Because it wipes the table before every load, it always reflects exactly what the source file contains — including any revisions to historical values. The downside is that it rewrites the entire table every time, which is wasteful if only a few rows changed.

**Incremental** is the most efficient: it only writes new rows, so it's fast and doesn't duplicate data. But it has a blind spot — it only looks at whether a date is new, not whether an existing value changed. In this CPI example, PCPI25M2.csv contains revised values for dates already in the table, and the incremental method silently ignores them.

For CPI data specifically, **truncate & load** is the most appropriate method because historical revisions matter. Incremental works well for data that is append-only and never revised (e.g. transaction logs, sensor readings).